In [1]:
import numpy as np

np.random.seed(42)

Importar os dados

In [3]:
# Importar dados brutos

import pandas as pd

ANOS = [2023, 2024, 2025]
AREAS = {1: "area1", 2: "area2"}

ARQ_DEMANDA = {
    2023: {1: "dados_brutos/area1_2023.csv", 2: "dados_brutos/area2_2023.csv"},
    2024: {1: "dados_brutos/area1_2024.csv", 2: "dados_brutos/area2_2024.csv"},
    2025: {1: "dados_brutos/area1_2025_completo.csv", 2: "dados_brutos/area2_2025_completo.csv"},
}

ARQ_TEMP = {
    2023: "dados_brutos/INMET_SE_SP_A711_SAO CARLOS_01-01-2023_A_31-12-2023.CSV",
    2024: "dados_brutos/INMET_SE_SP_A711_SAO CARLOS_01-01-2024_A_31-12-2024.CSV",
    2025: "dados_brutos/INMET_SE_SP_A711_SAO CARLOS_01-01-2025_A_31-12-2025.CSV",
}

dados_demanda = {
    (ano, area): pd.read_csv(ARQ_DEMANDA[ano][area], encoding="latin-1", sep=";")
    for ano in ANOS
    for area in AREAS
}

dados_temp = {
    ano: pd.read_csv(ARQ_TEMP[ano], encoding="latin-1", sep=";", skiprows=8)
    for ano in ANOS
}

data2023a1, data2024a1, data2025a1 = (dados_demanda[(ano, 1)] for ano in ANOS)
data2023a2, data2024a2, data2025a2 = (dados_demanda[(ano, 2)] for ano in ANOS)
data2023temp, data2024temp, data2025temp = (dados_temp[ano] for ano in ANOS)

datacardapio = pd.read_csv('dados_brutos/cardapio.txt', delimiter=',')
datacardapio.to_csv('cardapio.csv', index=False)

agrupamento = pd.ExcelFile('dados_brutos/agrupamento_cardapio.xlsx')

Tratamento dos arquivos de demanda

In [4]:
# Padronizar a grafia dos dias da semana

for _df in (data2023a1, data2024a1, data2025a1, data2023a2, data2024a2, data2025a2):
    _df['Dia_semana'] = _df['Dia_semana'].str.lower()

In [5]:
def transformar(df, ano, area):

    if area == 1:
        df = df[['Mês', 'Dia_semana', 'Dia_mês', 'Total_almoço', 'Total_jantar']]

        df_long = df.melt(
            id_vars=['Mês', 'Dia_semana', 'Dia_mês'],
            value_vars=['Total_almoço', 'Total_jantar'],
            var_name='refeicao',
            value_name='total'
        )

        df_long['refeicao'] = df_long['refeicao'].replace({
            'Total_almoço': 'almoco',
            'Total_jantar': 'jantar'
        })

        ordem = ['almoco', 'jantar']
        df_long['refeicao'] = pd.Categorical(
            df_long['refeicao'],
            categories=ordem,
            ordered=True
        )

    else:  # área 2

        df = df[['Mês', 'Dia_semana', 'Dia_mês', 'Total_almoço']]
        df_long = df.rename(columns={'Total_almoço': 'total'})
        df_long['refeicao'] = 'almoco'

    # adiciona coluna ano
    df_long.insert(0, 'Ano', ano)

    # ordenação
    df_long = df_long.sort_values(
        by=['Ano', 'Mês', 'Dia_mês', 'refeicao', 'total']
    ).reset_index(drop=True)

    df_long = df_long[['Ano', 'Mês', 'Dia_semana', 'Dia_mês', 'refeicao', 'total']]
    return df_long

In [6]:
data2023a1 = transformar(data2023a1,2023,1)
data2024a1 = transformar(data2024a1,2024,1)
data2025a1 = transformar(data2025a1,2025,1)

data2023a2 = transformar(data2023a2,2023,2)
data2024a2= transformar(data2024a2,2024,2)
data2025a2= transformar(data2025a2,2025,2)

Tratamento dos arquivos de informações meteorológicas

In [7]:
def processa_temp(df):

    # Remove colunas que não serão usadas
    df = df.drop(columns=[
        'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
        'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
        'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
        'RADIACAO GLOBAL (Kj/m²)',
        'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)',
        'TEMPERATURA DO PONTO DE ORVALHO (°C)',
        'TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)',
        'TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)',
        'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
        'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
        'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
        'Unnamed: 19'
    ], errors='ignore')

    # Cria datetime
    df['datetime'] = pd.to_datetime(
        df['Data'] + ' ' + df['Hora UTC'],
        format='%Y/%m/%d %H%M UTC',
        errors='coerce'
    )

    # Extrai data e hora
    df['data'] = df['datetime'].dt.date
    df['hora'] = df['datetime'].dt.hour

    # Colunas numéricas
    colunas_numericas = [
        'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
        'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)',
        'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)',
        'UMIDADE RELATIVA DO AR, HORARIA (%)',
        'VENTO, VELOCIDADE HORARIA (m/s)',
        'VENTO, RAJADA MAXIMA (m/s)'
    ]

    # Converte para numérico
    for col in colunas_numericas:
        df[col] = pd.to_numeric(
            df[col].astype(str).str.replace(',', '.', regex=False),
            errors='coerce'
        )

    # Renomeia colunas
    df = df.rename(columns={
        'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)': 'Precipitacao_mm',
        'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)': 'Temp_max_C',
        'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)': 'Temp_min_C',
        'UMIDADE RELATIVA DO AR, HORARIA (%)': 'Umid_rel_ar',
        'VENTO, RAJADA MAXIMA (m/s)': 'Vento_rajada_maxima_ms',
        'VENTO, VELOCIDADE HORARIA (m/s)': 'Vento_velocidade_ms'
    })

    # Atualiza lista de colunas
    colunas_numericas = [
        'Precipitacao_mm',
        'Temp_max_C',
        'Temp_min_C',
        'Umid_rel_ar',
        'Vento_velocidade_ms',
        'Vento_rajada_maxima_ms'
    ]

    # Classifica refeição
    def classifica_refeicao(h):
        if 10 <= h <= 11:
            return 'cafe da manha'
        elif 14 <= h <= 16:
            return 'almoco'
        elif 20 <= h <= 22:
            return 'jantar'
        else:
            return None

    df['refeicao'] = df['hora'].apply(classifica_refeicao)

    # Mantém só refeições desejadas
    df = df[df['refeicao'].notna()]

    # Agrupa
    df_refeicao = (
        df.groupby(['data', 'refeicao'])[colunas_numericas]
        .mean()
        .reset_index()
    )

    # Data → Ano/Mês/Dia
    df_refeicao['data'] = pd.to_datetime(df_refeicao['data'])
    df_refeicao['Ano'] = df_refeicao['data'].dt.year
    df_refeicao['Mês'] = df_refeicao['data'].dt.month
    df_refeicao['Dia_mês'] = df_refeicao['data'].dt.day

    # Reorganiza colunas
    df_refeicao = df_refeicao[['Ano','Mês','Dia_mês','refeicao'] + colunas_numericas]

    ordem_refeicao = ['cafe da manha','almoco','jantar']
    df_refeicao['refeicao'] = pd.Categorical(df_refeicao['refeicao'], categories=ordem_refeicao, ordered=True)

    df_refeicao = df_refeicao.sort_values(by=['Ano','Mês','Dia_mês','refeicao']).reset_index(drop=True)

    return df_refeicao

In [8]:
# Aplica a função
data2023_meteorologia = processa_temp(data2023temp)
data2024_meteorologia = processa_temp(data2024temp)
data2025_meteorologia = processa_temp(data2025temp)

Tratamento dos dados de calendário

In [9]:
from calendario_utils import gerar_calendario

data_calendario = gerar_calendario(data_inicio='2023-01-01', data_fim='2025-12-31')

Tratamentos de dados do cardápio

In [10]:
import pandas as pd

datacardapio[['sobremesa_1', 'sobremesa_2']] = datacardapio['sobremesa'].str.split(' e ', n=1, expand=True)

datacardapio = datacardapio.drop(columns=['sobremesa'])
datacardapio['data'] = datacardapio['data'].str.strip()

datacardapio['data'] = pd.to_datetime(datacardapio['data'], errors='coerce')  # erros viram NaT

datacardapio['Ano'] = datacardapio['data'].dt.year
datacardapio['Mês'] = datacardapio['data'].dt.month
datacardapio['Dia_mês'] = datacardapio['data'].dt.day

datacardapio = datacardapio.drop(columns=['data'])

datacardapio = datacardapio[['Ano', 'Mês', 'Dia_mês', 'refeicao', 'prato_principal_1',
                             'prato_principal_2', 'guarnição', 'sobremesa_1', 'sobremesa_2']]

datacardapio = datacardapio.drop(columns=['sobremesa_2'])

### Limpeza de qualidade de dados do cardápio

In [11]:
# Remove linhas do cardápio com "Erro de leitura" em qualquer campo de prato
colunas_prato = ['prato_principal_1', 'prato_principal_2', 'guarnição', 'sobremesa_1']
linhas_antes = len(datacardapio)
mask_erro_leitura = (datacardapio[colunas_prato] == 'Erro de leitura').any(axis=1)
datacardapio = datacardapio[~mask_erro_leitura].copy()
print(f"Cardápio: removidas {mask_erro_leitura.sum()} linha(s) com 'Erro de leitura' "
      f"({linhas_antes} -> {len(datacardapio)})")

# Remove o dia 11/10/2023 (dia com problema conhecido de coleta/registro),
# aplicado uma única vez para todas as áreas.
DIA_PROBLEMATICO = {'Ano': 2023, 'Mês': 10, 'Dia_mês': 11}

def remover_dia(df, dia=DIA_PROBLEMATICO, label=''):
    antes = len(df)
    mask = (
        (df['Ano'] == dia['Ano']) &
        (df['Mês'] == dia['Mês']) &
        (df['Dia_mês'] == dia['Dia_mês'])
    )
    df_limpo = df[~mask].copy()
    print(f"{label}: removidas {mask.sum()} linha(s) do dia {dia['Dia_mês']}/{dia['Mês']}/{dia['Ano']} "
          f"({antes} -> {len(df_limpo)})")
    return df_limpo

data2023a1 = remover_dia(data2023a1, label='demanda área 1 (2023)')
data2023a2 = remover_dia(data2023a2, label='demanda área 2 (2023)')

Cardápio: removidas 0 linha(s) com 'Erro de leitura' (1666 -> 1666)
demanda área 1 (2023): removidas 2 linha(s) do dia 11/10/2023 (712 -> 710)
demanda área 2 (2023): removidas 1 linha(s) do dia 11/10/2023 (302 -> 301)


Unindo dfs

In [12]:
def checar_merge(df_esquerda, df_direita, resultado, nome, on, how='inner'):
    print(f"[merge: {nome}] esquerda={len(df_esquerda)} direita={len(df_direita)} "
          f"-> resultado={len(resultado)} (how={how}, on={on})")
    if how == 'inner' and len(resultado) > max(len(df_esquerda), len(df_direita)):
        raise AssertionError(
            f"Merge '{nome}' duplicou linhas inesperadamente "
            f"(resultado maior que ambas as entradas). Verifique chaves duplicadas."
        )
    if len(resultado) < 0.9 * len(df_esquerda):
        print(f"  ATENÇÃO: '{nome}' perdeu mais de 10% das linhas da esquerda "
              f"({len(df_esquerda)} -> {len(resultado)}). Confirme se isso é esperado.")

In [13]:
# Juntar anos da demanda

data_demandaa1 = pd.concat([data2023a1, data2024a1, data2025a1], ignore_index=True, sort=False)
data_demandaa2 = pd.concat([data2023a2, data2024a2, data2025a2], ignore_index=True, sort=False)

# Reorganizando para colocar 'Ano' como primeira coluna
colunas = ['Ano'] + [c for c in data_demandaa1.columns if c != 'Ano']
dataarea1 = data_demandaa1[colunas]

In [14]:
# Juntar demanda e cardápio

df1a1 = pd.merge(data_demandaa1, datacardapio, on=['Ano', 'Mês', 'Dia_mês', 'refeicao'])
checar_merge(data_demandaa1, datacardapio, df1a1, 'demanda x cardápio (área 1)',
             on=['Ano', 'Mês', 'Dia_mês', 'refeicao'])

df1a2 = pd.merge(data_demandaa2, datacardapio, on=['Ano', 'Mês', 'Dia_mês', 'refeicao'])
checar_merge(data_demandaa2, datacardapio, df1a2, 'demanda x cardápio (área 2)',
             on=['Ano', 'Mês', 'Dia_mês', 'refeicao'])

[merge: demanda x cardápio (área 1)] esquerda=2172 direita=1666 -> resultado=1515 (how=inner, on=['Ano', 'Mês', 'Dia_mês', 'refeicao'])
  ATENÇÃO: 'demanda x cardápio (área 1)' perdeu mais de 10% das linhas da esquerda (2172 -> 1515). Confirme se isso é esperado.
[merge: demanda x cardápio (área 2)] esquerda=974 direita=1666 -> resultado=743 (how=inner, on=['Ano', 'Mês', 'Dia_mês', 'refeicao'])
  ATENÇÃO: 'demanda x cardápio (área 2)' perdeu mais de 10% das linhas da esquerda (974 -> 743). Confirme se isso é esperado.


In [15]:
# Juntar anos metereológicos

data_met = pd.concat([data2023_meteorologia, data2024_meteorologia, data2025_meteorologia], ignore_index=True, sort=False)

In [16]:
# Juntar dados metereológicos

df2a1 = pd.merge(df1a1, data_met, on=['Ano', 'Mês', 'Dia_mês', 'refeicao'])
checar_merge(df1a1, data_met, df2a1, 'cardápio x meteorologia (área 1)',
             on=['Ano', 'Mês', 'Dia_mês', 'refeicao'])

df2a2 = pd.merge(df1a2, data_met, on=['Ano', 'Mês', 'Dia_mês', 'refeicao'])
checar_merge(df1a2, data_met, df2a2, 'cardápio x meteorologia (área 2)',
             on=['Ano', 'Mês', 'Dia_mês', 'refeicao'])

[merge: cardápio x meteorologia (área 1)] esquerda=1515 direita=3288 -> resultado=1515 (how=inner, on=['Ano', 'Mês', 'Dia_mês', 'refeicao'])
[merge: cardápio x meteorologia (área 2)] esquerda=743 direita=3288 -> resultado=743 (how=inner, on=['Ano', 'Mês', 'Dia_mês', 'refeicao'])


In [17]:
# Juntar calendario

df_finala1 = pd.merge(df2a1, data_calendario, on=['Ano', 'Mês', 'Dia_mês','Dia_semana'])
checar_merge(df2a1, data_calendario, df_finala1, 'dados x calendário (área 1)',
             on=['Ano', 'Mês', 'Dia_mês', 'Dia_semana'])

df_finala2 = pd.merge(df2a2, data_calendario, on=['Ano', 'Mês', 'Dia_mês','Dia_semana'])
checar_merge(df2a2, data_calendario, df_finala2, 'dados x calendário (área 2)',
             on=['Ano', 'Mês', 'Dia_mês', 'Dia_semana'])

[merge: dados x calendário (área 1)] esquerda=1515 direita=1096 -> resultado=1515 (how=inner, on=['Ano', 'Mês', 'Dia_mês', 'Dia_semana'])
[merge: dados x calendário (área 2)] esquerda=743 direita=1096 -> resultado=741 (how=inner, on=['Ano', 'Mês', 'Dia_mês', 'Dia_semana'])


In [18]:
# Agrupar cardápio

def agrupar_nomes(data, xlsx, sheet_name, column_in_data):
  data_agrupamento = pd.read_excel(xlsx, sheet_name, header=(0))

  # Transpor para ficar mais fácil de tratar
  data_agrupamento_transposed = data_agrupamento.T

  #Itere pela coluna, pegue todos os valores dela
  grupos = []

  for name, col in data_agrupamento_transposed.items():
    #Remove os nans
    categoria_variacoes = list(filter(lambda x: not pd.isna(x), col.values))
    grupos.append(categoria_variacoes)


  # -----------------------
  # Substitui nomes

  # Criar dicionário
  substituicoes = {}

  #Aplicando a substituição
  for array_categoria in grupos:
    substituicoes[array_categoria[0]] = array_categoria


  for prato_padrao, variacoes in substituicoes.items():
      data[column_in_data] = data[column_in_data].replace(variacoes, prato_padrao)

  return data

In [19]:
# Agrupar cardápio

df_finala1 = agrupar_nomes(df_finala1, agrupamento, 'Carnes', 'prato_principal_1')
df_finala1 = agrupar_nomes(df_finala1, agrupamento, 'Vegetariano', 'prato_principal_2')
df_finala1 = agrupar_nomes(df_finala1, agrupamento, 'Guarnição', 'guarnição')
df_finala1 = agrupar_nomes(df_finala1, agrupamento, 'Sobremesa doce', 'sobremesa_1')

df_finala2 = agrupar_nomes(df_finala2, agrupamento, 'Carnes', 'prato_principal_1')
df_finala2 = agrupar_nomes(df_finala2, agrupamento, 'Vegetariano', 'prato_principal_2')
df_finala2 = agrupar_nomes(df_finala2, agrupamento, 'Guarnição', 'guarnição')
df_finala2 = agrupar_nomes(df_finala2, agrupamento, 'Sobremesa doce', 'sobremesa_1')

### Diagnóstico antes do `dropna()`

In [20]:
def diagnosticar_nulos(df, nome):
    nulos_por_coluna = df.isna().sum()
    nulos_por_coluna = nulos_por_coluna[nulos_por_coluna > 0]
    linhas_com_nulo = df.isna().any(axis=1).sum()
    print(f"[{nome}] {linhas_com_nulo} de {len(df)} linhas têm ao menos um valor nulo "
          f"({linhas_com_nulo / len(df):.1%})")
    if not nulos_por_coluna.empty:
        print(nulos_por_coluna.to_string())
    else:
        print("  (nenhuma coluna com nulos)")

diagnosticar_nulos(df_finala1, 'df_finala1 (antes do dropna)')
diagnosticar_nulos(df_finala2, 'df_finala2 (antes do dropna)')

df_finala1 = df_finala1.dropna()
df_finala2 = df_finala2.dropna()
df_finala2 = df_finala2.drop(columns=['refeicao'])

[df_finala1 (antes do dropna)] 82 de 1515 linhas têm ao menos um valor nulo (5.4%)
Precipitacao_mm           82
Temp_max_C                82
Temp_min_C                82
Umid_rel_ar               82
Vento_velocidade_ms       82
Vento_rajada_maxima_ms    82
[df_finala2 (antes do dropna)] 60 de 741 linhas têm ao menos um valor nulo (8.1%)
total                     11
Precipitacao_mm           49
Temp_max_C                49
Temp_min_C                49
Umid_rel_ar               49
Vento_velocidade_ms       49
Vento_rajada_maxima_ms    49


### Verificações de sanidade (sanity checks)

In [21]:
def checar_sanidade(df, nome, colunas_chave):
    print(f"--- Checagens de sanidade: {nome} ---")

    duplicadas = df.duplicated(subset=colunas_chave).sum()
    print(f"Linhas duplicadas em {colunas_chave}: {duplicadas}")

    if (df['total'] < 0).any():
        print(f"  ATENÇÃO: {(df['total'] < 0).sum()} linha(s) com 'total' negativo")

    if 'Precipitacao_mm' in df.columns and (df['Precipitacao_mm'] < 0).any():
        print(f"  ATENÇÃO: {(df['Precipitacao_mm'] < 0).sum()} linha(s) com precipitação negativa")

    if {'Temp_min_C', 'Temp_max_C'}.issubset(df.columns):
        inconsistente = df['Temp_min_C'] > df['Temp_max_C']
        if inconsistente.any():
            print(f"  ATENÇÃO: {inconsistente.sum()} linha(s) com Temp_min_C > Temp_max_C")

    if 'Umid_rel_ar' in df.columns:
        fora_faixa = ~df['Umid_rel_ar'].between(0, 100)
        if fora_faixa.any():
            print(f"  ATENÇÃO: {fora_faixa.sum()} linha(s) com Umid_rel_ar fora de [0, 100]")

    print(f"Linhas finais: {len(df)}")
    print()

cols_chave_a1 = ['Ano', 'Mês', 'Dia_mês', 'refeicao']
cols_chave_a2 = ['Ano', 'Mês', 'Dia_mês']

checar_sanidade(df_finala1, 'dataarea1', cols_chave_a1)
checar_sanidade(df_finala2, 'dataarea2', cols_chave_a2)

--- Checagens de sanidade: dataarea1 ---
Linhas duplicadas em ['Ano', 'Mês', 'Dia_mês', 'refeicao']: 5
Linhas finais: 1433

--- Checagens de sanidade: dataarea2 ---
Linhas duplicadas em ['Ano', 'Mês', 'Dia_mês']: 5
Linhas finais: 681



In [22]:
df_finala1.to_csv('../../data/dataarea1.csv', index=False)
df_finala2.to_csv('../../data/dataarea2.csv', index=False)

In [23]:
import os

if os.path.exists('../data_treatment_completo/cardapio.csv'):
    os.remove('../data_treatment_completo/cardapio.csv')